# Lab 2 — Number Detection (YOLOv8)

## §1. Clone / Pull + path setup

In [ ]:
import os
import sys
from pathlib import Path

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import userdata  # type: ignore
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Ma-XD/ITMO-CV.git"
    
    if not Path("/content/ITMO-CV").exists():
        !git clone {repo_url} /content/ITMO-CV
    else:
        %cd /content/ITMO-CV
        !git pull
    
    sys.path.insert(0, "/content/ITMO-CV/lab2-DET")
    %cd /content/ITMO-CV/lab2-DET
else:
    LAB_DIR = Path(".").resolve()
    if LAB_DIR.name != "lab2-DET":
        %cd lab2-DET
    if str(LAB_DIR) not in sys.path:
        sys.path.insert(0, str(LAB_DIR))

## §2. Drive mount

In [ ]:
from env_config import is_colab

if is_colab:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')

## §3. Зависимости

In [ ]:
!pip install -q -r requirements.txt

## §4. Verify окружения

In [ ]:
from env_config import print_env
print_env()

## §5. Загрузка датасета
Копируем заранее подготовленные архивы с Google Drive и распаковываем их в локальное хранилище Colab (`/content/data/`).

In [ ]:
%%bash
set -e

DATA_DIR=/content/data
mkdir -p "$DATA_DIR"

# 1. SVHN
SVHN_SRC=/content/drive/MyDrive/ITMO-CV/lab2-DET/data/svhn_yolo.zip
if [ -d "$DATA_DIR/svhn_yolo" ] && [ -f "$DATA_DIR/svhn_yolo/data.yaml" ]; then
  echo "✅ SVHN уже распакован"
else
  [ ! -f "$SVHN_SRC" ] && echo "❌ Архив не найден: $SVHN_SRC" && exit 1
  echo "📥 Копируем SVHN..."
  cp "$SVHN_SRC" /tmp/svhn.zip
  echo "📦 Распаковываем SVHN..."
  unzip -q /tmp/svhn.zip -d "$DATA_DIR"
  # Убеждаемся, что пути в data.yaml абсолютные, чтобы YOLO не искал их относительно текущей папки
  sed -i "s|^path: .*|path: $DATA_DIR/svhn_yolo|g" "$DATA_DIR/svhn_yolo/data.yaml"
  rm /tmp/svhn.zip
fi
echo "   Изображений SVHN (train): $(find "$DATA_DIR/svhn_yolo/images/train" -name '*.png' 2>/dev/null | wc -l)"

echo "----------------------------------------"

# 2. Toronto Number Detection
TORONTO_SRC=/content/drive/MyDrive/ITMO-CV/lab2-DET/data/numberdetection.v1i.yolov8.zip
if [ -d "$DATA_DIR/numberdetection" ] && [ -f "$DATA_DIR/numberdetection/data.yaml" ]; then
  echo "✅ Toronto Number Detection уже распакован"
else
  [ ! -f "$TORONTO_SRC" ] && echo "❌ Архив не найден: $TORONTO_SRC" && exit 1
  echo "📥 Копируем Toronto..."
  cp "$TORONTO_SRC" /tmp/toronto.zip
  echo "📦 Распаковываем Toronto..."
  unzip -q /tmp/toronto.zip -d "$DATA_DIR/numberdetection"
  # Исправляем пути в data.yaml для Toronto
  sed -i "s|^path: .*|path: $DATA_DIR/numberdetection|g" "$DATA_DIR/numberdetection/data.yaml" || echo "path: $DATA_DIR/numberdetection" >> "$DATA_DIR/numberdetection/data.yaml"
  rm /tmp/toronto.zip
fi
echo "   Изображений Toronto (train): $(find \"$DATA_DIR/numberdetection/train/images\" -name '*.jpg' 2>/dev/null | wc -l)"

echo "----------------------------------------"

# 3. Street Photos
STREET_SRC=/content/drive/MyDrive/ITMO-CV/lab2-DET/data/street_photos
if [ -d "$DATA_DIR/street_photos" ]; then
  echo "✅ Street Photos уже скопированы"
else
  if [ -d "$STREET_SRC" ]; then
    echo "📥 Копируем Street Photos..."
    cp -r "$STREET_SRC" "$DATA_DIR/"
  else
    echo "⚠️ Папка $STREET_SRC не найдена на Drive. Создайте её и положите туда свои фото."
  fi
fi
echo "   Фотографий для инференса: $(find "$DATA_DIR/street_photos" -name '*.png' -o -name '*.jpg' -o -name '*.jpeg' 2>/dev/null | wc -l)"


## §6. Подготовка разметки (YOLO format)

Исходный датасет SVHN поставляется в формате `digitStruct.mat` (HDF5). Для работы с YOLOv8 я предварительно сконвертировал его в нужный формат с помощью скрипта `scripts/build_dataset.py`.

Скрипт выполнил:
1. Парсинг `digitStruct.mat` и извлечение координат боксов (top, left, height, width).
2. Перевод координат в нормализованный формат YOLO (cx, cy, w, h).
3. Разделение на `train` (90%) и `val` (10%).
4. Создание файла `data.yaml`.

В Colab этот шаг пропускается, так как уже загружен готовый архив `svhn_yolo.zip`.

## §7. Работа с датасетом
Проверим корректность разметки: загрузим одну картинку из SVHN и нарисуем поверх неё боксы из соответствующего `.txt` файла.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from config import SVHN_DIR

def plot_yolo_sample(img_path: Path, label_path: Path):
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img)
    
    if label_path.exists():
        with open(label_path, "r") as f:
            lines = f.readlines()
        
        w, h = img.size
        for line in lines:
            class_id, cx, cy, bw, bh = map(float, line.strip().split())
            # YOLO format (cx, cy, bw, bh) -> Matplotlib format (x_min, y_min, width, height)
            box_w = bw * w
            box_h = bh * h
            x_min = (cx * w) - (box_w / 2)
            y_min = (cy * h) - (box_h / 2)
            
            rect = patches.Rectangle((x_min, y_min), box_w, box_h, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)
            ax.text(x_min, y_min - 2, str(int(class_id)), color='red', fontsize=12, weight='bold')
    else:
        print("⚠️ Файл разметки не найден!")
        
    plt.axis('off')
    plt.show()

# Визуализируем один пример из обучающей выборки SVHN
sample_img = SVHN_DIR / "images/train/1.png"
sample_lbl = SVHN_DIR / "labels/train/1.txt"

if sample_img.exists():
    print(f"Визуализация: {sample_img.name}")
    plot_yolo_sample(sample_img, sample_lbl)
else:
    print("⚠️ Изображение не найдено. Проверьте пути.")

## §8. Train: YOLOv8n on SVHN (Direct)
**Цель**: Получить базовую точку отсчёта (baseline) — насколько хорошо детектируются цифры (10 классов), если обучать YOLOv8n «с нуля» (от COCO-весов) сразу на SVHN, без предобучения на Toronto.

Перед экспериментами фиксируем общие гиперпараметры в словаре `COMMON_TRAIN_ARGS`, чтобы переиспользовать их во всех запусках.

In [ ]:
from ultralytics import YOLO
from config import LOG_DIR, SVHN_YAML, TORONTO_YAML

# Общие гиперпараметры для всех экспериментов
COMMON_TRAIN_ARGS = {
    "imgsz": 640,        # Стандартный размер входа для YOLOv8
    "batch": 16,         # Размер батча (подходит для Colab T4)
    "optimizer": "AdamW",# Современный оптимизатор, отлично подходит для fine-tuning
    "project": LOG_DIR,  # Сохраняем логи и чекпоинты сразу в нашу папку (на Drive)
    "device": 0,         # Использовать GPU 0
    "plots": True,       # Генерировать графики метрик
    "save": True,        # Обязательно сохранять чекпоинты (.pt)
    "save_period": 5,    # Сохранять чекпоинт каждые 5 эпох (на случай обрыва Colab)
    "exist_ok": True,    # Перезаписывать папку при перезапуске ячейки
    "fliplr": 0.0        # ОТКЛЮЧАЕМ горизонтальное отзеркаливание (важно для цифр!)
}

print(f"📁 Чекпоинты будут сохраняться в: {LOG_DIR}")

In [ ]:
# Эксперимент 1: Обучение базовой модели на SVHN
model_direct = YOLO("yolov8n.pt")

results_direct = model_direct.train(
    data=SVHN_YAML,
    name="svhn_direct",
    epochs=50,
    **COMMON_TRAIN_ARGS
)

## §9. Train: YOLOv8n on Toronto (Pre-training)
**Цель**: «Прогреть» модель на более общем датасете надписей/номеров от University of Toronto (1 класс), чтобы Backbone научился выделять контрастный текст в городской среде.

In [ ]:
# Эксперимент 2: Предобучение на Toronto
model_pretrain = YOLO("yolov8n.pt")

results_pretrain = model_pretrain.train(
    data=TORONTO_YAML,
    name="toronto_pretrain",
    epochs=30,
    **COMMON_TRAIN_ARGS
)

## §10. Train: YOLOv8n on SVHN (Fine-tuning)
**Цель**: Дообучить модель, уже знакомую с задачей детекции номеров (после Toronto), на целевом датасете SVHN.

**Архитектурный нюанс**: Так как Toronto имеет 1 класс, а SVHN — 10 классов, YOLO автоматически сбросит Detection Head, но сохранит веса Backbone. Ожидание — mAP вырастет относительно бейзлайна из пункта 8.

In [ ]:
# Эксперимент 3: Дообучение на SVHN с весами от Toronto
toronto_best_weights = LOG_DIR / "toronto_pretrain" / "weights" / "best.pt"

if not toronto_best_weights.exists():
    print(f"⚠️ Чекпоинт {toronto_best_weights} не найден. Сначала запустите §9.")
else:
    model_finetuned = YOLO(toronto_best_weights)
    
    results_finetuned = model_finetuned.train(
        data=SVHN_YAML,
        name="svhn_finetuned",
        epochs=50,
        **COMMON_TRAIN_ARGS
    )

## §11. Запуск инференса (ручной)
**Manual IoU Demo**: Берем одну картинку из тестовой выборки SVHN, прогоняем через лучшую модель (end-to-end), рисуем предсказанные (красные) и Ground Truth (зеленые) боксы. Вручную считаем и выводим метрику **Intersection over Union (IoU)** для каждой пары, чтобы наглядно показать понимание метрики детекции.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
from config import SVHN_DIR, LOG_DIR

def calculate_iou(box1, box2):
    # box: [x_min, y_min, x_max, y_max]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0

# 1. Загружаем лучшую модель
best_model_path = LOG_DIR / "svhn_finetuned" / "weights" / "best.pt"
if not best_model_path.exists():
    print(f"⚠️ Модель {best_model_path} еще не обучена. Используем базовую yolov8n.pt для демо.")
    model = YOLO("yolov8n.pt")
    model_name_for_title = "yolov8n.pt (Baseline)"
else:
    print(f"✅ Загружена лучшая модель: {best_model_path.parent.parent.name}")
    model = YOLO(best_model_path)
    model_name_for_title = "svhn_finetuned"

# 2. Берем 3 случайные картинки из test
test_images = list((SVHN_DIR / "images/test").glob("*.png"))
sample_paths = random.sample(test_images, 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
if not isinstance(axes, np.ndarray):
    axes = [axes]

for ax, sample_img_path in zip(axes, sample_paths):
    sample_lbl_path = SVHN_DIR / "labels/test" / (sample_img_path.stem + ".txt")
    
    img = Image.open(sample_img_path)
    w, h = img.size
    
    # 3. Читаем Ground Truth (зеленые боксы)
    gt_boxes = []
    if sample_lbl_path.exists():
        with open(sample_lbl_path, "r") as f:
            for line in f:
                cls_id, cx, cy, bw, bh = map(float, line.strip().split())
                x_min = (cx - bw/2) * w
                y_min = (cy - bh/2) * h
                x_max = (cx + bw/2) * w
                y_max = (cy + bh/2) * h
                gt_boxes.append([x_min, y_min, x_max, y_max])
    
    # 4. Получаем предсказания (красные боксы)
    results = model.predict(sample_img_path, conf=0.25, verbose=False)
    pred_boxes = []
    for box in results[0].boxes.xyxy: # xyxy format (x_min, y_min, x_max, y_max)
        pred_boxes.append(box.tolist())
    
    # 5. Визуализация и расчет IoU
    ax.imshow(img)
    
    # Рисуем GT
    for box in gt_boxes:
        rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                 linewidth=2, edgecolor='green', facecolor='none', linestyle='--')
        ax.add_patch(rect)
    
    # Рисуем Pred и считаем IoU с лучшим GT
    for p_box in pred_boxes:
        rect = patches.Rectangle((p_box[0], p_box[1]), p_box[2]-p_box[0], p_box[3]-p_box[1], 
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        # Ищем максимальный IoU с GT
        best_iou = 0
        for g_box in gt_boxes:
            iou = calculate_iou(p_box, g_box)
            if iou > best_iou:
                best_iou = iou
                
        # Выводим IoU над боксом
        ax.text(p_box[0], p_box[1] - 5, f"IoU: {best_iou:.2f}", color='red', fontsize=10, weight='bold', backgroundcolor='white')
    
    ax.axis('off')
    ax.set_title(f"{sample_img_path.name}")

# Легенда на последнем графике
axes[-1].plot([], [], color='green', linestyle='--', label='Ground Truth')
axes[-1].plot([], [], color='red', label='Prediction')
axes[-1].legend(loc='upper right')

plt.suptitle(f"Manual IoU Demo | Model: {model_name_for_title}", fontsize=16)
plt.tight_layout()
plt.show()


## §12. Инференс и оценка на test
Прогон обученных моделей на тестовой выборке SVHN. Согласно заданию, нам необходимо оценить качество детекции с помощью метрик mAP, Precision, Recall и убедиться, что mAP >= 0.6.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
from config import LOG_DIR, SVHN_YAML

print("🚀 Оценка модели svhn_direct (Baseline)...")
model_direct_path = LOG_DIR / "svhn_direct" / "weights" / "best.pt"
if model_direct_path.exists():
    model_direct = YOLO(model_direct_path)
    # plots=True создаст PR_curve.png и другие графики в папке runs/val
    metrics_direct = model_direct.val(data=SVHN_YAML, split="test", plots=True, verbose=False)
    print(f"Baseline mAP50-95: {metrics_direct.box.map:.4f}")
    print(f"Baseline mAP50:    {metrics_direct.box.map50:.4f}")
    print(f"Baseline Precision:{metrics_direct.box.mp:.4f}")
    print(f"Baseline Recall:   {metrics_direct.box.mr:.4f}")
else:
    print("⚠️ Модель svhn_direct не найдена.")
    metrics_direct = None

print("\n🚀 Оценка модели svhn_finetuned (Pre-trained on Toronto)...")
model_finetuned_path = LOG_DIR / "svhn_finetuned" / "weights" / "best.pt"
if model_finetuned_path.exists():
    model_finetuned = YOLO(model_finetuned_path)
    metrics_finetuned = model_finetuned.val(data=SVHN_YAML, split="test", plots=True, verbose=False)
    print(f"Finetuned mAP50-95: {metrics_finetuned.box.map:.4f}")
    print(f"Finetuned mAP50:    {metrics_finetuned.box.map50:.4f}")
    print(f"Finetuned Precision:{metrics_finetuned.box.mp:.4f}")
    print(f"Finetuned Recall:   {metrics_finetuned.box.mr:.4f}")
    
    # Выводим PR-кривую для лучшей модели
    pr_curve_path = Path(metrics_finetuned.save_dir) / "PR_curve.png"
    if pr_curve_path.exists():
        print("\n📊 PR-кривая (Finetuned):")
        display(Image(filename=str(pr_curve_path), width=600))
else:
    print("⚠️ Модель svhn_finetuned не найдена.")
    metrics_finetuned = None


## §13. Сравнение моделей
Сведем результаты в единую таблицу для наглядного сравнения.

In [ ]:
import pandas as pd
from IPython.display import display

results_data = []

if metrics_direct is not None:
    results_data.append({
        "Model": "SVHN Direct (Baseline)",
        "mAP50-95": metrics_direct.box.map,
        "mAP50": metrics_direct.box.map50,
        "Precision": metrics_direct.box.mp,
        "Recall": metrics_direct.box.mr
    })

if metrics_finetuned is not None:
    results_data.append({
        "Model": "SVHN Finetuned (Toronto pre-train)",
        "mAP50-95": metrics_finetuned.box.map,
        "mAP50": metrics_finetuned.box.map50,
        "Precision": metrics_finetuned.box.mp,
        "Recall": metrics_finetuned.box.mr
    })

if results_data:
    df = pd.DataFrame(results_data)
    df = df.set_index("Model")
    display(df.style.format("{:.4f}").background_gradient(cmap="Greens", axis=0))
else:
    print("⚠️ Нет данных для сравнения. Сначала обучите модели.")


## §14. Real-World Inference (Street Photos)
Прогон лучшей модели на пользовательских уличных фотографиях. Так как наша модель обучена детектировать отдельные цифры, мы применим простую эвристику (пост-процессинг): объединим близко стоящие по горизонтали боксы цифр в единый прямоугольник, чтобы выделить номер целиком.

In [ ]:
import cv2
import numpy as np
from config import STREET_PHOTOS_DIR

def group_digits_into_plates(boxes, max_dist_x=50, max_dist_y=20):
    """
    Группирует боксы отдельных цифр в боксы номеров.
    boxes: список [x_min, y_min, x_max, y_max]
    """
    if not boxes:
        return []
        
    # Сортируем боксы слева направо
    boxes = sorted(boxes, key=lambda b: b[0])
    
    plates = []
    current_plate = list(boxes[0])
    
    for box in boxes[1:]:
        # Проверяем расстояние между текущим номером и новой цифрой
        dist_x = box[0] - current_plate[2]
        dist_y = abs(box[1] - current_plate[1])
        
        if dist_x < max_dist_x and dist_y < max_dist_y:
            # Расширяем текущий номер
            current_plate[0] = min(current_plate[0], box[0])
            current_plate[1] = min(current_plate[1], box[1])
            current_plate[2] = max(current_plate[2], box[2])
            current_plate[3] = max(current_plate[3], box[3])
        else:
            # Начинаем новый номер
            plates.append(current_plate)
            current_plate = list(box)
            
    plates.append(current_plate)
    return plates

if not STREET_PHOTOS_DIR.exists():
    print(f"⚠️ Папка {STREET_PHOTOS_DIR} не найдена.")
else:
    street_images = list(STREET_PHOTOS_DIR.glob("*.jpg")) + list(STREET_PHOTOS_DIR.glob("*.png"))
    
    if not street_images:
        print("⚠️ В папке нет фотографий.")
    else:
        print(f"📸 Найдено фотографий: {len(street_images)}")
        
        # Используем лучшую модель
        if 'model_finetuned' in locals():
            infer_model = model_finetuned
            print("Используем модель: svhn_finetuned")
        else:
            infer_model = YOLO("yolov8n.pt")
            print("Используем модель: yolov8n.pt (Baseline)")

        fig, axes = plt.subplots(len(street_images), 1, figsize=(10, 8 * len(street_images)))
        if len(street_images) == 1:
            axes = [axes]
            
        for ax, img_path in zip(axes, street_images):
            img = Image.open(img_path)
            ax.imshow(img)
            
            # Предикт
            results = infer_model.predict(img_path, conf=0.25, verbose=False)
            
            # Собираем все предсказанные боксы цифр
            digit_boxes = []
            for box in results[0].boxes.xyxy:
                digit_boxes.append(box.tolist())
                
            # Группируем в номера
            plate_boxes = group_digits_into_plates(digit_boxes)
            
            # Рисуем финальные рамки номеров
            for p_box in plate_boxes:
                rect = patches.Rectangle((p_box[0], p_box[1]), p_box[2]-p_box[0], p_box[3]-p_box[1], 
                                         linewidth=3, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                ax.text(p_box[0], p_box[1] - 10, "Detected Number", color='white', fontsize=12, weight='bold', backgroundcolor='blue')
                
            ax.axis('off')
            ax.set_title(img_path.name)
            
        plt.tight_layout()
        plt.show()
